# Sentence-level GPT-2 XL surprisal

Word surprisal = mean over the word's GPT-2 sub-tokens of −ln P(sub-token | preceding text), in nats.
Sentence surprisal = mean of its word surprisals.

By default this reads the example output in `data/examples/sentences/output`. To score your own list, set
`STIMULI` and `RECOMPUTE = True`. You need GPT-2 XL first (`linguistic fetch gpt2-xl`), and a GPU is much faster.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

STIMULI = ROOT / "data/examples/sentences/stimuli.csv"
OUT = ROOT / "data/examples/sentences/output"
RECOMPUTE = False

if RECOMPUTE:
    from linguistic.cli import _run
    OUT = ROOT / "out" / STIMULI.parent.name
    _run(STIMULI, OUT, metric_spec=["lm_gpt2-xl"])

words = pd.read_csv(OUT / "words.csv", keep_default_na=False, na_values=[""])
cols = ["stimulus_id", "word_idx", "word", "lm_n_subtokens_gpt2xl", "lm_surprisal_gpt2xl", "lm_entropy_gpt2xl"]
words[cols].head(12)

,stimulus_id,word_idx,word,lm_n_subtokens_gpt2xl,lm_surprisal_gpt2xl,lm_entropy_gpt2xl
0,harvard_list1_01,1,The,1.0,2.687234,6.763116
1,harvard_list1_01,2,birch,2.0,7.492240,8.411595
2,harvard_list1_01,3,canoe,1.0,7.001229,4.991343
3,harvard_list1_01,4,slid,1.0,9.353335,4.790022
4,harvard_list1_01,5,on,1.0,3.899148,3.837498
5,harvard_list1_01,6,the,1.0,1.214134,3.014382
6,harvard_list1_01,7,smooth,1.0,4.725453,4.686790
7,harvard_list1_01,8,planks,2.0,4.618503,4.550695
8,harvard_list1_02,1,Glue,2.0,6.596195,6.763116
9,harvard_list1_02,2,the,1.0,3.401780,6.738591


## Sentence level: mean of word surprisals

In [2]:
sent = (words.groupby("stimulus_id", sort=False)
             .agg(text=("word", " ".join), n_words=("word", "size"),
                  surprisal_gpt2xl=("lm_surprisal_gpt2xl", "mean"),
                  entropy_gpt2xl=("lm_entropy_gpt2xl", "mean"))
             .reset_index())

# the same value the pipeline writes to stimuli.csv
stim = pd.read_csv(OUT / "stimuli.csv")
assert (sent.set_index("stimulus_id")["surprisal_gpt2xl"]
        - stim.set_index("stimulus_id")["lm_surprisal_gpt2xl"]).abs().max() < 1e-9

sent.sort_values("surprisal_gpt2xl")

,stimulus_id,text,n_words,surprisal_gpt2xl,entropy_gpt2xl
2,harvard_list1_03,It's easy to tell the depth of a well,9,3.536500,4.477694
14,harvard_list2_05,Help the woman get back to her feet,8,3.866838,4.897231
10,harvard_list2_01,The boy was there when the sun rose,8,3.981421,5.439947
3,harvard_list1_04,These days a chicken leg is a rare dish,9,4.278364,4.759149
4,harvard_list1_05,Rice is often served in round bowls,7,4.312484,4.862242
1,harvard_list1_02,Glue the sheet to the dark blue background,8,4.412569,5.434763
15,harvard_list2_06,A pot of tea helps to pass the evening,9,4.532112,4.826673
12,harvard_list2_03,The source of the huge river is the clear spring,10,4.772488,5.950608
9,harvard_list1_10,A large size in stockings is hard to sell,9,4.783430,5.182503
13,harvard_list2_04,Kick the ball straight and follow through,7,4.926771,4.612441


In [3]:
sent[["surprisal_gpt2xl", "entropy_gpt2xl", "n_words"]].describe().round(2)

,surprisal_gpt2xl,entropy_gpt2xl,n_words
count,20.00,20.00,20.00
mean,4.99,5.15,7.95
std,0.84,0.42,0.94
min,3.54,4.48,6.00
25%,4.39,4.81,7.00
50%,4.95,5.12,8.00
75%,5.46,5.45,8.25
max,6.47,5.95,10.00


## Examples: highest and lowest surprisal

In [4]:
def show(df, label):
    print(label)
    for r in df.itertuples():
        print(f"  {r.surprisal_gpt2xl:.2f} nats  {r.text}")

ranked = sent.sort_values("surprisal_gpt2xl", ascending=False)
show(ranked.head(2), "High surprisal")
show(ranked.tail(2)[::-1], "Low surprisal")

High surprisal
  6.47 nats  Smoky fires lack flame and heat
  6.41 nats  The girl at the booth sold fifty bonds
Low surprisal
  3.54 nats  It's easy to tell the depth of a well
  3.87 nats  Help the woman get back to her feet


Each word counts equally, however many sub-tokens it has. If you instead pool all sub-tokens of the
sentence (sum of −ln P over sub-tokens ÷ number of sub-tokens, which is log-perplexity), long words that split into
many sub-tokens get more weight.